# Base Corrigida e Modelagem - Soja & Clima

Este notebook aplica as correções levantadas na análise exploratória (EDA) para gerar a base final de modelagem.
As correções incluem:
1. Limpeza física de sensores (temperaturas impossíveis, chuvas extremas).
2. Agregação por Ano-Safra (Out a Mar) para refletir o ciclo da soja.
3. Normalização pelo número de dias válidos (evitando viés de estações com falhas).
4. Filtro de área mínima para remover ruídos de pequenos produtores.
5. Corte temporal (a partir de 2009) para remover anos com alta cegueira climática.
6. Detrending (remoção da tendência tecnológica) para isolar o efeito do clima.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")

print("Carregando dados...")
df_clima = pd.read_parquet(PROCESSED_DIR / "clima_diario.parquet")
df_soja = pd.read_parquet(PROCESSED_DIR / "soja_microrregiao.parquet")
df_solo = pd.read_parquet(PROCESSED_DIR / "solo_microrregioes_limpo.parquet")
df_pacifico = pd.read_parquet(PROCESSED_DIR / "pacifico_indices.parquet")
mapa = pd.read_csv(PROCESSED_DIR / "estacao_microrregiao.csv")

Carregando dados...


### 1. Correções no Clima (Limpeza, Ano-Safra e Normalização)
- Removemos dados fisicamente impossíveis (Alerta 6).
- Recalculamos o clima apenas para a janela da soja Out-Mar (Alerta 5).
- Normalizamos as variáveis pelo `n_dias_validos` para evitar a correlação espúria de +0.74 (Alerta 2).

In [4]:
print("Aplicando limpeza física nos dados diários...")
df_clima = df_clima[
    (df_clima['temp_min'] >= -10) & 
    (df_clima['prec_total'] <= 300) & 
    (df_clima['gdd'] <= 35)
].copy()

print("Mapeando estações para microrregiões...")
mapa = mapa.dropna(subset=['CD_MICRO']).copy()
mapa['CD_MICRO'] = mapa['CD_MICRO'].astype(float).astype(int).astype(str)
mapa['codigo_estacao'] = mapa['codigo_estacao'].astype(str)
df_clima['codigo_estacao'] = df_clima['codigo_estacao'].astype(str)

df_clima_micro = df_clima.merge(mapa[['codigo_estacao', 'CD_MICRO']], on='codigo_estacao')

print("Recortando a janela da safra (Out-Mar)...")
df_clima_micro['data'] = pd.to_datetime(df_clima_micro['data'])
df_clima_micro['mes'] = df_clima_micro['data'].dt.month
df_clima_micro['ano_safra'] = np.where(df_clima_micro['mes'] >= 10, df_clima_micro['data'].dt.year + 1, df_clima_micro['data'].dt.year)

janela_safra = df_clima_micro[df_clima_micro['mes'].isin([10, 11, 12, 1, 2, 3])]

print("Agregando clima por safra e normalizando...")
clima_safra = janela_safra.groupby(['CD_MICRO', 'ano_safra']).agg(
    precipitacao_total=('prec_total', 'sum'),
    temp_media_anual=('temp_media', 'mean'),
    temp_max_anual=('temp_max', 'max'),
    temp_min_anual=('temp_min', 'min'),
    gdd_total=('gdd', 'sum'),
    dias_secos_total=('dia_seco', 'sum'),
    n_dias_validos=('prec_total', 'count')
).reset_index().rename(columns={'ano_safra': 'ano'})

clima_safra['taxa_chuva_diaria'] = clima_safra['precipitacao_total'] / clima_safra['n_dias_validos']
clima_safra['fracao_dias_secos'] = clima_safra['dias_secos_total'] / clima_safra['n_dias_validos']
clima_safra['gdd_medio_diario'] = clima_safra['gdd_total'] / clima_safra['n_dias_validos']

Aplicando limpeza física nos dados diários...
Mapeando estações para microrregiões...
Recortando a janela da safra (Out-Mar)...
Agregando clima por safra e normalizando...


### 2. Correções no Pacífico
- Agregamos o ONI e SOI apenas para a janela da safra (Out-Mar).

In [5]:
print("Agregando ONI/SOI por safra...")
df_pacifico['mes'] = df_pacifico['mes'].astype(int)
df_pacifico['ano'] = df_pacifico['ano'].astype(int)
df_pacifico['ano_safra'] = np.where(df_pacifico['mes'] >= 10, df_pacifico['ano'] + 1, df_pacifico['ano'])

pacifico_safra = df_pacifico[df_pacifico['mes'].isin([10, 11, 12, 1, 2, 3])]
pacifico_anual = pacifico_safra.groupby('ano_safra').agg(
    oni_medio=('oni', 'mean'),
    soi_medio=('soi', 'mean')
).reset_index().rename(columns={'ano_safra': 'ano'})

Agregando ONI/SOI por safra...


### 3. Consolidação e Filtros da Soja
- Filtramos áreas menores que 100 hectares para remover ruído (Alerta 7).
- Consolidamos Soja + Clima + Solo + Pacífico.
- Salvamos a base `base_final_corrigida.csv` (versão completa, ainda sem o corte temporal).

In [9]:
print("Consolidando a base final corrigida...")
df_soja['CD_MICRO'] = df_soja['CD_MICRO'].astype(float).astype(int).astype(str)
df_solo['CD_MICRO'] = df_solo['CD_MICRO'].astype(float).astype(int).astype(str)

# Filtra área mínima de 100 hectares para evitar ruído (Alerta 7)
df_soja = df_soja[df_soja['área_colhida'] >= 100].copy()

# NOVO: Filtra rendimentos fisicamente impossíveis para a soja comercial (Alerta 7)
df_soja = df_soja[df_soja['rendimento_recalculado'] <= 6000].copy()

df_final = df_soja.merge(clima_safra, on=['CD_MICRO', 'ano'], how='left')
df_final = df_final.merge(df_solo, on='CD_MICRO', how='left')
df_final = df_final.merge(pacifico_anual, on='ano', how='left')

df_final.to_csv(PROCESSED_DIR / "base_final_corrigida.csv", index=False)
print(f"✅ Base salva: base_final_corrigida.csv ({df_final.shape[0]} linhas)")

Consolidando a base final corrigida...
✅ Base salva: base_final_corrigida.csv (6359 linhas)


In [12]:
print("Removendo variáveis de solo redundantes (Multicolinearidade)...")

# Lista de colunas de solo que VAMOS MANTER (camada 0-5cm e as principais)
cols_solo_manter = [
    'phh2o_0-5cm_mean',  # pH
    'clay_0-5cm_mean',   # Argila (textura)
    'sand_0-5cm_mean',   # Areia (textura)
    'soc_0-5cm_mean',    # Carbono Orgânico
    'cec_0-5cm_mean',    # Capacidade de Troca Catiônica
    'bdod_0-5cm_mean',   # Densidade do solo
    'wv0033_0-5cm_mean'  # Água disponível
]

# Filtra o DataFrame final para manter apenas essas colunas + as outras do clima/soja
# (Identifica quais colunas são de solo para dropar as que não estão na lista)
cols_solo_todas = [c for c in df_final.columns if any(x in c for x in ['clay', 'sand', 'silt', 'bdod', 'soc', 'cec', 'phh2o', 'wv0033'])]
cols_solo_dropar = [c for c in cols_solo_todas if c not in cols_solo_manter]

df_final = df_final.drop(columns=cols_solo_dropar)
print(f"✅ Removidas {len(cols_solo_dropar)} colunas de solo redundantes.")
print(f"Colunas de solo mantidas: {cols_solo_manter}")

Removendo variáveis de solo redundantes (Multicolinearidade)...
✅ Removidas 17 colunas de solo redundantes.
Colunas de solo mantidas: ['phh2o_0-5cm_mean', 'clay_0-5cm_mean', 'sand_0-5cm_mean', 'soc_0-5cm_mean', 'cec_0-5cm_mean', 'bdod_0-5cm_mean', 'wv0033_0-5cm_mean']


### 4. Corte Temporal (2009) e Detrending Final
- Removemos os anos anteriores a 2009, que possuem mais de 25% de dados climáticos ausentes (Alerta 4).
- Recalculamos o Detrending (Regressão Polinomial Grau 2) sobre a série filtrada para isolar o efeito do clima (Alertas 1, 3 e 9).
- Salvamos a base definitiva: `base_modelagem_final.csv`.

In [13]:
print("Aplicando corte temporal (ano >= 2009)...")
df_modelagem = df_final[df_final['ano'] >= 2009].copy()

print("Calculando Detrending com Média Ponderada (Real)...")

# AQUI ESTÁ A CORREÇÃO: Média ponderada em vez de média simples
media_nacional = df_modelagem.groupby('ano').apply(
    lambda x: pd.Series({
        'rendimento_medio': (x['quantidade_produzida'].sum() * 1000) / x['área_colhida'].sum()
    })
).reset_index()

# Ajusta a curva polinomial nos dados REAIS
X_ano = media_nacional['ano'].values.reshape(-1, 1)
y_rend = media_nacional['rendimento_medio'].values

poly = PolynomialFeatures(degree=2)
modelo = LinearRegression().fit(poly.fit_transform(X_ano), y_rend)
media_nacional['desvio_clima'] = media_nacional['rendimento_medio'] - modelo.predict(poly.fit_transform(X_ano))

# Traz o desvio calculado de volta para a base
df_modelagem = df_modelagem.drop(columns=['desvio_clima'], errors='ignore')
df_modelagem = df_modelagem.merge(media_nacional[['ano', 'desvio_clima']], on='ano', how='left')

df_modelagem.to_csv(PROCESSED_DIR / "base_modelagem_final.csv", index=False)
print(f"✅ Base de modelagem salva com sucesso!")
print(f"   Shape: {df_modelagem.shape[0]} linhas x {df_modelagem.shape[1]} colunas")
print(f"   Anos: {df_modelagem['ano'].min()} a {df_modelagem['ano'].max()}")

Aplicando corte temporal (ano >= 2009)...
Calculando Detrending com Média Ponderada (Real)...


/var/folders/j3/kppygfjs7xj9nbzbwjj9cjc80000gn/T/ipykernel_76054/1893952916.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  media_nacional = df_modelagem.groupby('ano').apply(


✅ Base de modelagem salva com sucesso!
   Shape: 4428 linhas x 28 colunas
   Anos: 2009 a 2025


### 5. [Espaço Reservado para Futuras Correções]
Adicione novas células abaixo caso queira testar algo novo (ex: remoção de outliers extremos, imputação de nulos com KNN, criação de features de interação, etc.).